# RT-DETR Knowledge Distillation — Phase 2A/2D Training (Colab Pro+ · A100)

End-to-end runner for the 23-run Phase 2A ablation and the seeded Phase 2D / 2E final runs.
All orchestration lives in `scripts/run_ablation.sh` / `scripts/run_final.sh`; this notebook just
wires the Colab environment to those scripts.

## Pipeline

| Cell | Purpose |
|------|---------|
| 1    | Clone repo (with submodules), install deps, GPU check |
| 2    | Smoke test — `pytest tests/` (must pass 32/32) |
| 3    | COCO 30K subset — cache on Drive, symlink locally |
| 4    | Download canonical RT-DETR teacher checkpoint (lyuwenyu) |
| 5    | Cross-architecture KD verification (must pass before launch) |
| 6    | Configure Phase 2A env vars |
| 7    | Launch Phase 2A — 23 runs, ~50–70 h on A100, resumable across session drops |
| 8    | Progress tracker — re-runnable, parses Drive `runs/` |
| 9    | Aggregate results → CSV + Markdown |
| 10   | TensorBoard (live monitoring) |
| 11   | Phase 2D — final runs on full COCO 118K (after Phase 2A complete) |
| 12   | Phase 2E — 3-seed mean ± std aggregate |

## Session-drop resilience

Colab Pro+ sessions are capped at 24 h. `scripts/run_ablation.sh` skips any run whose
`checkpoint_best.pth` + `eval.log` already exist on Drive, so re-launching after a drop
resumes from the next pending run without re-training what is already done.


## 1. Environment setup


In [ ]:
# Mount Drive and clone the repo (with the lyuwenyu/RT-DETR submodule).
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

REPO_URL    = 'https://github.com/umutonuryasar/rt-detr-kd'
REPO_DIR    = '/content/rt_detr_kd'
DRIVE_BASE  = '/content/drive/MyDrive/rt_detr_kd'
RUNS_2A     = f'{DRIVE_BASE}/runs_phase2a'
RUNS_2D     = f'{DRIVE_BASE}/runs_phase2d'
WEIGHTS_DIR = f'{DRIVE_BASE}/weights'
COCO_DRIVE  = f'{DRIVE_BASE}/coco'
COCO_LOCAL  = '/content/coco'

for d in [RUNS_2A, RUNS_2D, WEIGHTS_DIR, COCO_DRIVE]:
    os.makedirs(d, exist_ok=True)

# Clone (with --recurse-submodules) or pull. Submodule init is required for
# the canonical teacher (third_party/RT-DETR) — without it tools/verify_teacher_kd.py
# raises RuntimeError before doing anything else.
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --recurse-submodules
    !git -C {REPO_DIR} submodule update --init --recursive

%cd {REPO_DIR}

# Project dependencies. tensorboard is for cell 10; pycocotools for eval.
!pip install -q -r requirements.txt 2>/dev/null || pip install -q pyyaml pillow numpy scipy
!pip install -q pycocotools pytest tensorboard

!nvidia-smi | head -20
print('\nSetup complete.')


## 2. Smoke test — 32 unit tests

If any test fails, **stop here**: something in the repo state is broken and Phase 2A
would propagate the failure across 23 runs. Common causes: stale submodule (re-run cell 1),
torch / torchvision version skew, missing pycocotools.


In [ ]:
!python -m pytest tests/ -q


## 3. COCO 30K subset

Phase 2A trains on a 30K stratified subset of COCO train2017 and the full val2017
(5K images, for eval). The download runs once and caches to Drive; subsequent sessions
symlink for fast I/O (Drive has rate-limit problems with millions of small reads).

Phase 2D switches to full COCO 118K — that download is in cell 11.


In [ ]:
import os

def symlink_coco(src, dst):
    if os.path.islink(dst):
        print(f'Already symlinked: {dst} → {os.readlink(dst)}')
        return
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f'Symlinked {src} → {dst}')
    else:
        print(f'NOT FOUND on Drive: {src} — running downloader...')

subset_train = f'{COCO_DRIVE}/train2017_30k'
if not os.path.exists(subset_train):
    print('Downloading 30K subset to Drive (one-time, ~10 min)...')
    !bash scripts/download_coco_subset.sh {COCO_DRIVE}

symlink_coco(COCO_DRIVE, COCO_LOCAL)

# Sanity check
n_train = len(os.listdir(f'{COCO_LOCAL}/train2017_30k')) if os.path.isdir(f'{COCO_LOCAL}/train2017_30k') else 0
n_val   = len(os.listdir(f'{COCO_LOCAL}/val2017'))       if os.path.isdir(f'{COCO_LOCAL}/val2017')       else 0
print(f'\nTrain (30K subset): {n_train:,} images')
print(f'Val (full):         {n_val:,} images')


## 4. Canonical teacher checkpoint

We use the official RT-DETR R50 checkpoint from
[`lyuwenyu/storage`](https://github.com/lyuwenyu/storage/releases/tag/v0.1) — 53.1 mAP, ~42 M params.
This is the **teacher**; the student is our simplified architecture (see README §3.2).

Cached on Drive (~570 MB) so subsequent sessions skip the download.


In [ ]:
import os, subprocess

TEACHER_URL  = 'https://github.com/lyuwenyu/storage/releases/download/v0.1/rtdetr_r50vd_6x_coco_from_paddle.pth'
TEACHER_PATH = f'{WEIGHTS_DIR}/rtdetr_r50vd_6x_coco_from_paddle.pth'
EXPECTED_MB  = 500   # actual is ~570 MB; we floor at 500 to catch corrupt downloads

need_download = True
if os.path.exists(TEACHER_PATH):
    size_mb = os.path.getsize(TEACHER_PATH) / 1e6
    if size_mb >= EXPECTED_MB:
        print(f'Teacher already present: {TEACHER_PATH} ({size_mb:.1f} MB)')
        need_download = False
    else:
        print(f'Existing teacher is suspiciously small ({size_mb:.1f} MB) — re-downloading.')
        os.remove(TEACHER_PATH)

if need_download:
    print(f'Downloading {TEACHER_URL} → {TEACHER_PATH}')
    !wget -q --show-progress -O {TEACHER_PATH} {TEACHER_URL}
    size_mb = os.path.getsize(TEACHER_PATH) / 1e6
    assert size_mb >= EXPECTED_MB, f'Download incomplete ({size_mb:.1f} MB)'
    print(f'Downloaded {size_mb:.1f} MB')


## 5. Cross-architecture KD verification

Loads the canonical teacher + simplified student and runs every supported KD type
against a dummy batch. Catches dim/shape mismatches, missing weights, and broken
stubs **before** the 50-hour Phase 2A launch. Expected output ends with:

    ✅ All 7 KD types pass cross-arch verification.

(`query` is reported as SKIP — deformable-attention teachers do not expose dense
decoder query embeddings; expected behaviour, not a bug.)


In [ ]:
LYUWENYU_CFG = 'third_party/RT-DETR/rtdetr_pytorch/configs/rtdetr/rtdetr_r50vd_6x_coco.yml'

!python tools/verify_teacher_kd.py \
    --student-cfg configs/rtdetr_r18vd_coco.yml \
    --lyuwenyu-cfg {LYUWENYU_CFG} \
    --teacher-weights {TEACHER_PATH} \
    --input-size 640 \
    --batch-size 2


## 6. Phase 2A configuration

`scripts/run_ablation.sh` reads its teacher source and per-run overrides from environment
variables. We set them here so the launch cell stays a one-liner.

Key knobs:

- `TEACHER_SOURCE=lyuwenyu` — use the canonical R50 teacher (the B1 architecture decision).
- `TEACHER_MIN_MAP=0.45` — the script runs a 200-image mAP eval on the teacher before any
  training; aborts if mAP < 0.45. Catches a silently-broken teacher checkpoint *before*
  burning 50 A100-hours on garbage KD signal.
- `EXTRA_TRAIN_ARGS` — A100 has 40 GB; we override the RTX-3050-shaped defaults
  (`batch=4, img=512, accum=2`) with the canonical (`batch=16, img=640, accum=1`).


In [ ]:
import os

os.environ['TEACHER_SOURCE']   = 'lyuwenyu'
os.environ['LYUWENYU_CFG']     = LYUWENYU_CFG
os.environ['TEACHER_WEIGHTS']  = TEACHER_PATH
os.environ['TEACHER_MIN_MAP']  = '0.45'
os.environ['EXTRA_TRAIN_ARGS'] = '--batch-size 16 --img-size 640 --accumulate-steps 1'

for k in ['TEACHER_SOURCE', 'LYUWENYU_CFG', 'TEACHER_WEIGHTS',
          'TEACHER_MIN_MAP', 'EXTRA_TRAIN_ARGS']:
    print(f'  {k:<18} = {os.environ[k]}')


## 7. Phase 2A launch (23 runs, ~50–70 h on A100)

`run_ablation.sh` orchestrates all 23 configurations end-to-end (train → FPS bench → COCO eval).
Completed runs are skipped on re-launch, so if the Colab session drops mid-Phase you can just
re-run this cell next session and pick up where it left off.

**Sessions are capped at 24 h** — expect to re-launch 2–3 times.


In [ ]:
!bash scripts/run_ablation.sh {COCO_LOCAL} {RUNS_2A}


## 8. Progress tracker (re-runnable anytime)

Walks `RUNS_2A/` on Drive and reports which of the 23 runs are done, in-progress, or pending.
Re-run this cell anytime — it does not touch training state.


In [ ]:
import os, re
from pathlib import Path

RUN_TAGS = [
    'run00_baseline',
    'run01_logit_l0.5_t2',  'run02_logit_l0.5_t4',  'run03_logit_l0.5_t8',
    'run04_logit_l1.0_t2',  'run05_logit_l1.0_t4',  'run06_logit_l1.0_t8',
    'run07_feature_l0.5',   'run08_feature_l1.0',
    'run09_combined_l1.0_t4',
    'run10_encoder_only_l1.0',     'run11_attention_only_l1.0',
    'run12_feature_teacher_r34',   'run13_feature_teacher_r50',
    'run14_cwd_l1.0',              'run15_mgd_l1.0',
    'run16_query_kd_l1.0',
    'run17_stage_adaptive_cosine',
    'run18_stage_adaptive_linear',
    'run19_stage_adaptive_step',
    'run20_stage_adaptive_sigmoid',
    'run21_stage_adaptive_inv_cosine',
    'run22_baseline_2x_epochs',
]

_MAP_PAT = re.compile(r'AP@\[\.5:\.95\]\s*:\s*([0-9.]+)')
_FPS_PAT = re.compile(r'Mean FPS\s*:\s*([0-9.]+)')

def _parse(path, pat):
    if not os.path.exists(path):
        return None
    m = pat.findall(Path(path).read_text(errors='ignore'))
    return float(m[-1]) if m else None

def show_progress(runs_root=RUNS_2A):
    done = part = 0
    print(f'{"Run":<35} {"Status":<10} {"mAP":>7} {"FPS":>7}')
    print('-' * 64)
    for tag in RUN_TAGS:
        out = f'{runs_root}/{tag}'
        best = f'{out}/checkpoint_best.pth'
        last = f'{out}/checkpoint_last.pth'
        mAP  = _parse(f'{out}/eval.log', _MAP_PAT)
        fps  = _parse(f'{out}/fps.log',  _FPS_PAT)
        if os.path.exists(best) and mAP is not None:
            status, done = 'DONE', done + 1
        elif os.path.exists(last) or os.path.isdir(out):
            status, part = 'PARTIAL', part + 1
        else:
            status = 'pending'
        mAP_s = f'{mAP:.3f}' if mAP is not None else '—'
        fps_s = f'{fps:.1f}' if fps is not None else '—'
        sym = '✓' if status == 'DONE' else ('~' if status == 'PARTIAL' else ' ')
        print(f'{sym} {tag:<33} {status:<10} {mAP_s:>7} {fps_s:>7}')
    print('-' * 64)
    print(f'Completed {done}/{len(RUN_TAGS)} · in-progress {part} · pending {len(RUN_TAGS) - done - part}')

show_progress()


## 9. Aggregate results → CSV + Markdown

After Phase 2A completes, this generates the paper-ready table
(`results.csv`, `results.md`) by walking the runs directory and parsing each run's
`eval.log` + `fps.log`.


In [ ]:
!python tools/aggregate_results.py --runs-dir {RUNS_2A}

from IPython.display import Markdown, display
md_path = f'{RUNS_2A}/results.md'
if os.path.exists(md_path):
    display(Markdown(open(md_path).read()))


## 10. TensorBoard (live monitoring)

Compare train losses across all 23 runs in one dashboard. Useful while Phase 2A is running.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_2A}


## 11. Phase 2D — final paper runs (full COCO 118K, 72 epochs, 3 seeds)

Run this section **only after Phase 2A is complete** and you have selected the top configs.
`scripts/run_final.sh` orchestrates the selected runs across 3 seeds (42, 1337, 2025).

Edit the `FINAL_RUNS` list inside `scripts/run_final.sh` to include your Phase 2A winners.
Default includes: baseline, feature-KD (run08), CWD, MGD — plus TODOs for best logit + best novel.

Estimated A100 time: ~80–100 h for the standard 4-method × 3-seed grid.


In [ ]:
import os, subprocess, threading

# ── Step 1: Download full COCO to LOCAL /content (fast SSD, ~25 min) ──────────
full_train_local = f'{COCO_LOCAL}/train2017'
if not os.path.exists(full_train_local) or len(os.listdir(full_train_local)) < 100000:
    print('Downloading full COCO to local SSD...')
    !bash scripts/download_coco_full.sh {COCO_LOCAL}
else:
    print(f'Full COCO already local: {len(os.listdir(full_train_local)):,} train images.')

# ── Step 2: Kick off Drive sync in background BEFORE training starts ──────────
# rsync copies local → Drive while training runs on the local SSD.
# If the session drops, re-run this cell — rsync --ignore-existing skips done files.
def _sync_to_drive():
    for subdir in ['train2017', 'val2017', 'annotations']:
        src = f'{COCO_LOCAL}/{subdir}/'
        dst = f'{COCO_DRIVE}/{subdir}/'
        os.makedirs(dst, exist_ok=True)
        subprocess.run(
            ['rsync', '-a', '--ignore-existing', '--info=progress2', src, dst],
            check=False,  # don't crash training if rsync hits a Drive hiccup
        )
    print('[rsync] Drive sync complete.')

sync_thread = threading.Thread(target=_sync_to_drive, daemon=True)
sync_thread.start()
print('Drive sync started in background (daemon thread).')

# ── Step 3: Training starts immediately on local data ────────────────────────
# COCO_LOCAL symlink already points to the right place from cell 3.
# run_final.sh reads from COCO_LOCAL (SSD) — fast I/O, no Drive rate-limit issues.
!bash scripts/run_final.sh {COCO_LOCAL} {RUNS_2D}

# ── Step 4: Wait for sync to finish (optional — cell blocks here) ────────────
print('Training done. Waiting for Drive sync to finish...')
sync_thread.join(timeout=3600)  # max 1h extra wait
if sync_thread.is_alive():
    print('WARNING: rsync still running — re-run the sync manually if needed.')
else:
    print('Drive sync complete. Full COCO now on Drive for future sessions.')


## 12. Phase 2E — seed-aggregated results (mean ± std)

After Phase 2D completes, this collapses the per-seed rows (`run08_feature_l1.0_seed42`,
`_seed1337`, `_seed2025` → one row) and reports mean ± std for every metric — the final
paper table.


In [ ]:
!python tools/aggregate_results.py --runs-dir {RUNS_2D} --seed-aggregate

from IPython.display import Markdown, display
md_path = f'{RUNS_2D}/results.md'
if os.path.exists(md_path):
    display(Markdown(open(md_path).read()))
